# PREPROCESSING

## Install packages

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from src.config import (
    DATA_PATH, TARGET_COL, DROP_COLS,
    CATEGORICAL_COLS, NUMERICAL_COLS, TEST_SIZE, RANDOM_STATE
)

In [ ]:
def load_data(path=DATA_PATH):
    """Load the Nigeria loan default dataset."""
    # TODO: Read the CSV file and return a DataFrame
    # Hint: pd.read_csv(path)
    
    pass



In [ ]:
def clean_data(df):
    """
    Clean the dataset.
    Steps:
    1. Drop columns in DROP_COLS
    2. Fill missing numerical values with the column median
    3. Fill missing categorical values with 'Unknown'
    """
    df = df.copy()

    # TODO: Step 1 — drop columns in DROP_COLS
    # Hint: df = df.drop(columns=DROP_COLS, errors='ignore')

    # TODO: Step 2 — fill missing numerical values with median
    # Hint: for col in NUMERICAL_COLS:
    #           if col in df.columns:
    #               df[col] = df[col].fillna(df[col].median())

    # TODO: Step 3 — fill missing categorical values with 'Unknown'
    # Hint: for col in CATEGORICAL_COLS:
    #           if col in df.columns:
    #               df[col] = df[col].fillna('Unknown')

    return df


In [ ]:
def encode_features(df):
    """
    Encode categorical columns using LabelEncoder.
    Loop through CATEGORICAL_COLS and label encode each one.
    """
    df = df.copy()
    le = LabelEncoder()

    # TODO: Loop through CATEGORICAL_COLS and encode each one
    # Hint: for col in CATEGORICAL_COLS:
    #           if col in df.columns:
    #               df[col] = le.fit_transform(df[col].astype(str))

    return df


In [ ]:
def prepare_features(df):
    """
    Full pipeline — returns X and y ready for modelling.
    Steps:
    1. Call clean_data(df)
    2. Call encode_features(df)
    3. Separate X and y
    4. Split into train and test sets
    5. Return X_train, X_test, y_train, y_test
    """
    # TODO: Step 1 — clean
    # TODO: Step 2 — encode
    # TODO: Step 3 — X = df.drop(columns=[TARGET_COL])
    #                y = df[TARGET_COL]
    # TODO: Step 4 — train_test_split with stratify=y
    # TODO: Step 5 — return X_train, X_test, y_train, y_test
    pass

# TRAIN

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import lightgbm as lgb
from scipy.stats import randint, loguniform, uniform

from src.preprocessing import load_data, prepare_features
from src.config import MODEL_PATH, RANDOM_STATE

In [ ]:
def get_models_and_params(spw):
    """
    Define all models and their hyperparameter search spaces.
    spw = scale_pos_weight for handling class imbalance.

    TODO: Add LightGBM and one more model of your choice.
    The Decision Tree and Logistic Regression are provided as examples.
    """
    models = {
        'Decision Tree': (
            DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
            {
                'max_depth':         randint(3, 15),
                'min_samples_leaf':  randint(1, 20),
                'criterion':         ['gini', 'entropy']
            }
        ),
        'Logistic Regression': (
            LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
            {
                'C': loguniform(0.01, 100)
            }
        ),
        # TODO: Add Random Forest
        # 'Random Forest': (
        #     RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
        #     {
        #         'n_estimators': randint(100, 400),
        #         'max_depth':    randint(3, 12),
        #         'max_features': ['sqrt', 'log2']
        #     }
        # ),

        # TODO: Add XGBoost
        # 'XGBoost': (
        #     XGBClassifier(scale_pos_weight=spw, eval_metric='logloss', verbosity=0, random_state=RANDOM_STATE),
        #     {
        #         'n_estimators':  randint(100, 400),
        #         'learning_rate': loguniform(1e-3, 3e-1),
        #         'max_depth':     randint(2, 8)
        #     }
        # ),

        # TODO: Add LightGBM
        # 'LightGBM': (
        #     lgb.LGBMClassifier(scale_pos_weight=spw, random_state=RANDOM_STATE, verbose=-1),
        #     {
        #         'n_estimators':  randint(100, 400),
        #         'learning_rate': loguniform(1e-3, 3e-1),
        #         'num_leaves':    randint(20, 80)
        #     }
        # ),
    }
    return models

In [ ]:
def tune_and_compare(models, X_train, y_train, X_test, y_test, n_iter=20, cv=5):
    """
    Run RandomizedSearchCV for each model and return comparison results.
    """
    results = []
    best_models = {}

    for name, (model, params) in models.items():
        print(f'Tuning {name}...')

        # TODO: Create RandomizedSearchCV with n_iter, cv=5, scoring='roc_auc'
        # Hint: search = RandomizedSearchCV(
        #           estimator=model,
        #           param_distributions=params,
        #           n_iter=n_iter, cv=cv, scoring='roc_auc',
        #           random_state=RANDOM_STATE, n_jobs=-1
        #       )
        # TODO: Fit the search on X_train and y_train
        # TODO: Get predictions and compute roc_auc_score on X_test and y_test
        # TODO: Append results dict to results list
        # TODO: Store best model in best_models[name]

        pass

    results_df = pd.DataFrame(results).sort_values('Test ROC-AUC', ascending=False)
    return results_df, best_models

In [ ]:
def save_model(model, path=MODEL_PATH):
    """Save the best model to disk."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(model, f)
    print(f'Model saved: {path}')


if __name__ == '__main__':
    df = load_data()
    print(f'Data loaded: {df.shape}')

    X_train, X_test, y_train, y_test = prepare_features(df)
    print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

    neg = int((y_train == 0).sum())
    pos = int((y_train == 1).sum())
    spw = neg / pos
    print(f'scale_pos_weight: {spw:.2f}')

    models = get_models_and_params(spw)

    print('\nRunning RandomizedSearchCV...\n')
    results_df, best_models = tune_and_compare(models, X_train, y_train, X_test, y_test)

    print('\n--- Model Comparison ---')
    print(results_df[['Model', 'CV ROC-AUC', 'Test ROC-AUC', 'Recall (Default)']].to_string(index=False))

    best_name  = results_df.iloc[0]['Model']
    best_model = best_models[best_name]
    print(f'\nBest model: {best_name}')

    save_model(best_model)
    print('Training complete.')

# PREDICT

In [ ]:
import pickle
import pandas as pd
from src.preprocessing import clean_data, encode_features
from src.config import MODEL_PATH, TARGET_COL

In [ ]:
def load_model(path=MODEL_PATH):
    with open(path, 'rb') as f:
        return pickle.load(f)

In [ ]:
def align_columns(df, model):
    """Reorder columns to match model training order."""
    if hasattr(model, 'feature_names_in_'):
        cols = [c for c in model.feature_names_in_ if c in df.columns]
        return df[cols]
    return df

In [ ]:
def predict_single(model, input_dict):
    """
    Predict default risk for a single loan application.
    Returns dict with prediction (0/1), label (Repaid/Default), probability.
    """
    # TODO: pd.DataFrame([input_dict])
    # TODO: clean_data and encode_features
    # TODO: drop TARGET_COL if present
    # TODO: align_columns
    # TODO: model.predict and model.predict_proba
    # TODO: return {'prediction': int, 'label': str, 'probability': float}
    pass

In [ ]:
def predict_batch(model, df):
    """
    Predict default risk for a batch of loan applications.
    Returns original df with Prediction, Probability, Label columns added.
    """
    # TODO: copy and process df
    # TODO: align_columns
    # TODO: model.predict and model.predict_proba
    # TODO: add columns and return
    pass

# EVALUATE

In [ ]:
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score, roc_curve
)
from src.preprocessing import load_data, prepare_features
from src.config import MODEL_PATH

In [ ]:
def load_model(path=MODEL_PATH):
    with open(path, 'rb') as f:
        return pickle.load(f)

In [ ]:
def evaluate_model(model, X_test, y_test):
    """
    Evaluate the model on the test set.
    Steps:
    1. Get predictions and probabilities
    2. Print classification report with target_names=['Repaid', 'Default']
    3. Print ROC-AUC and Average Precision
    4. Return y_pred and y_prob
    """
    # TODO: y_pred = model.predict(X_test)
    # TODO: y_prob = model.predict_proba(X_test)[:, 1]
    # TODO: print(classification_report(y_test, y_pred, target_names=['Repaid', 'Default']))
    # TODO: print ROC-AUC and Average Precision
    # TODO: return y_pred, y_prob
    pass

In [ ]:
def plot_confusion_matrix(y_test, y_pred):
    """Plot confusion matrix heatmap."""
    # TODO: Use sns.heatmap on confusion_matrix(y_test, y_pred)
    # Labels: ['Repaid', 'Default']
    pass

In [ ]:
def plot_roc_curve(y_test, y_prob):
    """Plot ROC curve."""
    # TODO: Use roc_curve and roc_auc_score to plot
    pass


if __name__ == '__main__':
    df = load_data()
    _, X_test, _, y_test = prepare_features(df)
    model = load_model()
    y_pred, y_prob = evaluate_model(model, X_test, y_test)
    plot_confusion_matrix(y_test, y_pred)
    plot_roc_curve(y_test, y_prob)
    plt.show()

# FINAL ADDITIONS and notes for presentation

<!-- structure for presentation slides -->
<!-- results from code that goes onto slides -->
<!-- final presentation contents -->